# Callback Manager API

Core callback dispatchers and run managers used by LangChain to send lifecycle events to callback handlers.

This module provides:

* Synchronous and asynchronous callback managers.
* Bound managers for LLM, chain, tool, and retriever runs.
* Event-dispatch helpers for sync and async callback handlers.
* Context managers for grouping unrelated operations into one traced chain run.
* Configuration helpers for handlers, tags, metadata, debugging, and LangSmith tracing.
* Utilities for dispatching user-defined custom events.

# Callback Manager Hierarchy

```text
BaseCallbackManager
├── CallbackManager
│   └── CallbackManagerForChainGroup
└── AsyncCallbackManager
    └── AsyncCallbackManagerForChainGroup

BaseRunManager
├── RunManager
│   ├── ParentRunManager
│   │   ├── CallbackManagerForChainRun
│   │   ├── CallbackManagerForToolRun
│   │   └── CallbackManagerForRetrieverRun
│   └── CallbackManagerForLLMRun
└── AsyncRunManager
    ├── AsyncParentRunManager
    │   ├── AsyncCallbackManagerForChainRun
    │   ├── AsyncCallbackManagerForToolRun
    │   └── AsyncCallbackManagerForRetrieverRun
    └── AsyncCallbackManagerForLLMRun
```




# LangChain Callback Manager Reference

Developer-facing statements defined in `langchain_core.callbacks.manager`.

# `trace_as_chain_group`

Creates a synchronous callback-manager context for grouping multiple operations under one traced chain run.

## Syntax

```python
trace_as_chain_group(
    group_name: str, # Name assigned to the traced chain group
    callback_manager: CallbackManager | None = None, # Optional existing callback manager
    *,
    inputs: dict[str, Any] | None = None, # Inputs recorded for the group run
    project_name: str | None = None, # Optional LangSmith project name
    example_id: str | UUID | None = None, # Optional LangSmith example identifier
    run_id: UUID | None = None, # Optional group run identifier
    tags: list[str] | None = None, # Inheritable tags applied to grouped runs
    metadata: dict[str, Any] | None = None, # Inheritable metadata applied to grouped runs
) -> Generator[
    CallbackManagerForChainGroup,
    None,
    None,
] # Yield the chain-group callback manager
```

The group ends automatically unless `on_chain_end()` or `on_chain_error()` was already called.

---

# `atrace_as_chain_group`

Creates an asynchronous callback-manager context for grouping multiple operations under one traced chain run.

## Syntax

```python
async atrace_as_chain_group(
    group_name: str, # Name assigned to the traced chain group
    callback_manager: AsyncCallbackManager | None = None, # Optional existing async callback manager
    *,
    inputs: dict[str, Any] | None = None, # Inputs recorded for the group run
    project_name: str | None = None, # Optional LangSmith project name
    example_id: str | UUID | None = None, # Optional LangSmith example identifier
    run_id: UUID | None = None, # Optional group run identifier
    tags: list[str] | None = None, # Inheritable tags applied to grouped runs
    metadata: dict[str, Any] | None = None, # Inheritable metadata applied to grouped runs
) -> AsyncGenerator[
    AsyncCallbackManagerForChainGroup,
    None,
] # Yield the asynchronous chain-group manager
```

The group ends automatically unless `on_chain_end()` or `on_chain_error()` was already called.

In [ ]:
from typing import Any # Import Any for callback input and output values
from uuid import UUID # Import UUID for callback run identifiers

from langchain_core.callbacks import BaseCallbackHandler # Import the callback-handler base class
from langchain_core.callbacks.manager import ( # Import callback managers and chain-group contexts
    AsyncCallbackManager, # Import the asynchronous callback manager
    CallbackManager, # Import the synchronous callback manager
    atrace_as_chain_group, # Import the asynchronous chain-group context
    trace_as_chain_group, # Import the synchronous chain-group context
) # Finish importing callback utilities
from langchain_core.runnables import RunnableLambda # Import RunnableLambda


class PrintChainHandler(BaseCallbackHandler): # Create a callback handler that displays chain events
    def on_chain_start( # Handle the start of a chain run
        self, # Current callback-handler instance
        serialized: dict[str, Any] | None, # Serialized chain information
        inputs: dict[str, Any] | Any, # Input supplied to the chain
        *,
        run_id: UUID, # Identifier of the current run
        parent_run_id: UUID | None = None, # Identifier of the parent run
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        name: str = kwargs.get("name") or (serialized or {}).get("name", "chain") # Find the chain name

        print( # Display the chain-start event
            f"START: {name} | Input: {inputs} | Child: {parent_run_id is not None}"
        ) # Finish displaying the event

        return # Finish handling the start event

    def on_chain_end( # Handle the completion of a chain run
        self, # Current callback-handler instance
        outputs: dict[str, Any] | Any, # Output produced by the chain
        *,
        run_id: UUID, # Identifier of the completed run
        parent_run_id: UUID | None = None, # Identifier of the parent run
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print( # Display the chain-end event
            f"END | Output: {outputs} | Child: {parent_run_id is not None}"
        ) # Finish displaying the event

        return # Finish handling the end event


def add_one(number: int) -> int: # Define the first operation
    return number + 1 # Add one to the input


def double(number: int) -> int: # Define the second operation
    return number * 2 # Double the input


add_runnable: RunnableLambda = RunnableLambda(add_one).with_config( # Create the first named Runnable
    {"run_name": "add_one"} # Set the Runnable trace name
) # Finish configuring the first Runnable

double_runnable: RunnableLambda = RunnableLambda(double).with_config( # Create the second named Runnable
    {"run_name": "double"} # Set the Runnable trace name
) # Finish configuring the second Runnable

handler: PrintChainHandler = PrintChainHandler() # Create the callback handler


sync_callback_manager: CallbackManager = CallbackManager.configure( # Create a synchronous callback manager
    inheritable_callbacks=[handler], # Add the handler to the group and its child runs
) # Finish creating the synchronous manager

with trace_as_chain_group( # Create one synchronous parent chain group
    "number_processing_group", # Set the parent group name
    callback_manager=sync_callback_manager, # Supply the callback manager
    inputs={"number": 5}, # Record the group input
    tags=["sync-example"], # Add an inheritable tag
    metadata={"mode": "synchronous"}, # Add inheritable metadata
) as group_manager: # Receive the grouped callback manager
    first_result: int = add_runnable.invoke( # Execute the first child Runnable
        5, # Supply the initial input
        config={"callbacks": group_manager}, # Attach the group callbacks
    ) # Finish the first invocation

    final_result: int = double_runnable.invoke( # Execute the second child Runnable
        first_result, # Supply the first Runnable's output
        config={"callbacks": group_manager}, # Attach the same group callbacks
    ) # Finish the second invocation

    group_manager.on_chain_end( # Explicitly complete the parent group
        {"output": final_result} # Record the final group output
    ) # Finish the parent chain

print("Synchronous result:", final_result) # Display the synchronous result


async def run_async_group() -> int: # Define the asynchronous chain-group example
    async_callback_manager: AsyncCallbackManager = AsyncCallbackManager.configure( # Create an async callback manager
        inheritable_callbacks=[handler], # Add the handler to the group and its children
    ) # Finish creating the asynchronous manager

    async with atrace_as_chain_group( # Create one asynchronous parent chain group
        "async_number_processing_group", # Set the parent group name
        callback_manager=async_callback_manager, # Supply the async callback manager
        inputs={"number": 10}, # Record the group input
        tags=["async-example"], # Add an inheritable tag
        metadata={"mode": "asynchronous"}, # Add inheritable metadata
    ) as group_manager: # Receive the asynchronous group manager
        first_result: int = await add_runnable.ainvoke( # Execute the first child asynchronously
            10, # Supply the initial input
            config={"callbacks": group_manager}, # Attach the group callbacks
        ) # Finish the first asynchronous invocation

        final_result: int = await double_runnable.ainvoke( # Execute the second child asynchronously
            first_result, # Supply the first result
            config={"callbacks": group_manager}, # Attach the same group callbacks
        ) # Finish the second asynchronous invocation

        await group_manager.on_chain_end( # Explicitly complete the async parent group
            {"output": final_result} # Record the final group output
        ) # Finish the asynchronous parent chain

    return final_result # Return the final asynchronous result


async_result: int = await run_async_group() # Run this line directly in Jupyter

print("Asynchronous result:", async_result) # Display the asynchronous result

# `BaseRunManager: RunManagerMixin`

`BaseRunManager` stores callback state for one active run.

## Fields

```python
run_id: UUID # Identifier of the current run
handlers: list[BaseCallbackHandler] # Handlers active for the current run
inheritable_handlers: list[BaseCallbackHandler] # Handlers inherited by child runs
parent_run_id: UUID | None # Identifier of the parent run
tags: list[str] # Tags active for the current run
inheritable_tags: list[str] # Tags inherited by child runs
metadata: dict[str, Any] # Metadata active for the current run
inheritable_metadata: dict[str, Any] # Metadata inherited by child runs
```

## Constructor

```python
BaseRunManager(
    *,
    run_id: UUID, # Identifier of the current run
    handlers: list[BaseCallbackHandler], # Active callback handlers
    inheritable_handlers: list[BaseCallbackHandler], # Handlers inherited by child runs
    parent_run_id: UUID | None = None, # Optional parent run identifier
    tags: list[str] | None = None, # Active run tags
    inheritable_tags: list[str] | None = None, # Tags inherited by child runs
    metadata: dict[str, Any] | None = None, # Active run metadata
    inheritable_metadata: dict[str, Any] | None = None, # Metadata inherited by child runs
) -> None # Initialize the bound run manager
```

## Class Method

### `get_noop_manager`

Returns a manager with a generated run ID and no handlers, tags, or metadata.

In [ ]:
from uuid import UUID, uuid4 # Import UUID types and the UUID generator

from langchain_core.callbacks import BaseCallbackHandler # Import the callback-handler base class
from langchain_core.callbacks.manager import BaseRunManager # Import BaseRunManager


class DemoHandler(BaseCallbackHandler): # Define a simple callback handler
    pass # Use the default callback-handler behaviour


handler: DemoHandler = DemoHandler() # Create the callback handler

parent_id: UUID = uuid4() # Generate an identifier for the parent run

current_run_id: UUID = uuid4() # Generate an identifier for the current run

run_manager: BaseRunManager = BaseRunManager( # Create a manager for one active run
    run_id=current_run_id, # Set the current run identifier
    handlers=[handler], # Add a handler active for this run
    inheritable_handlers=[handler], # Add a handler inherited by child runs
    parent_run_id=parent_id, # Set the parent run identifier
    tags=["demo", "synchronous"], # Add tags active for this run
    inheritable_tags=["demo"], # Add tags inherited by child runs
    metadata={"operation": "text-processing"}, # Add metadata active for this run
    inheritable_metadata={"environment": "development"}, # Add metadata inherited by children
) # Finish creating the run manager

noop_manager: BaseRunManager = BaseRunManager.get_noop_manager() # Create a manager with no callbacks


print("Current run ID:", run_manager.run_id) # Display the current run identifier

print("Parent run ID:", run_manager.parent_run_id) # Display the parent run identifier

print("Handler count:", len(run_manager.handlers)) # Display the number of active handlers

print("Inheritable handler count:", len(run_manager.inheritable_handlers)) # Display inherited handler count

print("Tags:", run_manager.tags) # Display the active tags

print("Inheritable tags:", run_manager.inheritable_tags) # Display tags inherited by child runs

print("Metadata:", run_manager.metadata) # Display active metadata

print("Inheritable metadata:", run_manager.inheritable_metadata) # Display inherited metadata

print("No-op handler count:", len(noop_manager.handlers)) # Show that the no-op manager has no handlers

print("No-op tags:", noop_manager.tags) # Show that the no-op manager has no tags

print("No-op metadata:", noop_manager.metadata) # Show that the no-op manager has no metadata

# `RunManager: BaseRunManager`

`RunManager` dispatches synchronous events for one active run.

## Methods

### `on_text`

Dispatches text received during the run.

### `on_retry`

Dispatches a Tenacity retry event.

In [ ]:
from typing import Any # Import Any for additional callback arguments
from uuid import uuid4 # Import the UUID generator

from tenacity import ( # Import Tenacity retry utilities
    RetryCallState, # Import the retry-state type
    Retrying, # Import the synchronous retry controller
    retry_if_exception_type, # Import exception-based retry condition
    stop_after_attempt, # Import the maximum-attempt condition
    wait_fixed, # Import the fixed-delay strategy
) # Finish importing Tenacity utilities

from langchain_core.callbacks import BaseCallbackHandler # Import the callback-handler base class
from langchain_core.callbacks.manager import RunManager # Import the synchronous run manager


class PrintHandler(BaseCallbackHandler): # Create a handler for text and retry events
    def on_text( # Handle text dispatched by RunManager
        self, # Current callback-handler instance
        text: str, # Text received during the run
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Text event:", text) # Display the received text
        print("Tags:", kwargs.get("tags", [])) # Display the run tags
        return # Finish handling the text event

    def on_retry( # Handle a retry event dispatched by RunManager
        self, # Current callback-handler instance
        retry_state: RetryCallState, # Current Tenacity retry state
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Retry event: attempt", retry_state.attempt_number) # Display the failed attempt number
        return # Finish handling the retry event


attempts: dict[str, int] = {"count": 0} # Store the operation-attempt count

handler: PrintHandler = PrintHandler() # Create the callback handler

run_manager: RunManager = RunManager( # Create a manager for one active run
    run_id=uuid4(), # Generate the current run identifier
    handlers=[handler], # Add the active callback handler
    inheritable_handlers=[handler], # Allow child runs to inherit the handler
    tags=["retry-example"], # Add tags to callback events
    metadata={"operation": "doubling"}, # Add metadata to the active run
) # Finish creating the run manager


def unstable_operation(number: int) -> int: # Define an operation that temporarily fails
    attempts["count"] += 1 # Increase the operation-attempt count

    run_manager.on_text( # Dispatch progress text during the active run
        f"Executing operation attempt {attempts['count']}" # Supply the progress message
    ) # Finish dispatching the text event

    if attempts["count"] < 3: # Fail during the first two attempts
        raise ValueError("Temporary failure") # Raise a retryable exception

    return number * 2 # Return the successful result


retry_controller: Retrying = Retrying( # Configure synchronous retries
    retry=retry_if_exception_type(ValueError), # Retry only ValueError exceptions
    stop=stop_after_attempt(3), # Allow at most three total attempts
    wait=wait_fixed(0), # Retry immediately without waiting
    before_sleep=run_manager.on_retry, # Dispatch a retry event before another attempt
    reraise=True, # Raise the final exception when all attempts fail
) # Finish configuring retry behaviour

result: int = retry_controller(unstable_operation, 10) # Execute the operation with retries

print("Final result:", result) # Display the successful result

# `ParentRunManager: RunManager`

`ParentRunManager` can create callback managers for nested child runs.

## Method

### `get_child`

Returns a `CallbackManager` containing inherited handlers, tags, and metadata.

An optional tag is added only to the child manager.

In [ ]:
from uuid import uuid4 # Import the UUID generator

from langchain_core.callbacks import BaseCallbackHandler # Import the callback-handler base class
from langchain_core.callbacks.manager import CallbackManager, ParentRunManager # Import parent and child managers


class DemoHandler(BaseCallbackHandler): # Create a simple callback handler
    pass # Use the default handler behaviour


handler: DemoHandler = DemoHandler() # Create the callback handler

parent_manager: ParentRunManager = ParentRunManager( # Create the parent run manager
    run_id=uuid4(), # Generate the parent run identifier
    handlers=[handler], # Add a handler to the parent run
    inheritable_handlers=[handler], # Allow child runs to inherit the handler
    tags=["parent-only"], # Add a tag active only on the parent
    inheritable_tags=["shared-tag"], # Add a tag inherited by child runs
    metadata={"parent": True}, # Add parent-only metadata
    inheritable_metadata={"environment": "development"}, # Add inherited metadata
) # Finish creating the parent manager

child_manager: CallbackManager = parent_manager.get_child( # Create a nested child callback manager
    tag="child-only", # Add a tag only to the child manager
) # Finish creating the child manager

print("Child parent ID:", child_manager.parent_run_id) # Display the parent run identifier

print("Child handler count:", len(child_manager.handlers)) # Display inherited handler count

print("Child tags:", child_manager.tags) # Display inherited and child-only tags

print("Child metadata:", child_manager.metadata) # Display inherited metadata

# `AsyncRunManager: BaseRunManager, ABC`

`AsyncRunManager` is the abstract asynchronous equivalent of `RunManager`.

## Abstract Method

### `get_sync`

Returns the synchronous run-manager equivalent.

## Methods

### `on_text`

Asynchronously dispatches text received during the run.

### `on_retry`

Asynchronously dispatches a Tenacity retry event.

# `AsyncParentRunManager: AsyncRunManager`

`AsyncParentRunManager` creates asynchronous callback managers for nested child runs.

It remains abstract until `get_sync()` is implemented by a concrete subclass.

## Method

### `get_child`

Returns an `AsyncCallbackManager` containing inherited handlers, tags, and metadata.

# `CallbackManagerForLLMRun: RunManager, LLMManagerMixin`

Synchronous callback manager bound to one LLM or chat-model run.

## Methods

### `on_llm_new_token`

Dispatches a generated token or structured content block and its optional generation chunk.

### `on_llm_end`

Dispatches the completed `LLMResult`.

### `on_llm_error`

Dispatches an exception raised during generation.

### `on_stream_event`

Dispatches one protocol event produced by `stream_events(version="v3")`.

In [ ]:
from typing import Any # Import Any for callback arguments and protocol-event data
from uuid import UUID, uuid4 # Import UUID types and generator

from langchain_core.callbacks import BaseCallbackHandler # Import the callback-handler base class
from langchain_core.callbacks.manager import CallbackManagerForLLMRun # Import the LLM run manager
from langchain_core.outputs import Generation, GenerationChunk, LLMResult # Import LLM output types


class PrintLLMHandler(BaseCallbackHandler): # Create a handler for LLM events
    def on_llm_new_token( # Handle each generated token
        self, # Current handler instance
        token: str, # Newly generated token
        *,
        run_id: UUID, # Current LLM run identifier
        chunk: GenerationChunk | None = None, # Optional generation chunk
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("New token:", token) # Display the token
        return # Finish handling the token

    def on_llm_end( # Handle successful LLM completion
        self, # Current handler instance
        response: LLMResult, # Completed LLM result
        *,
        run_id: UUID, # Current LLM run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Final text:", response.generations[0][0].text) # Display the final generated text
        return # Finish handling completion

    def on_llm_error( # Handle an LLM execution error
        self, # Current handler instance
        error: BaseException, # Exception raised during generation
        *,
        run_id: UUID, # Current LLM run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("LLM error:", error) # Display the exception
        return # Finish handling the error

    def on_stream_event( # Handle one version-3 protocol event
        self, # Current handler instance
        event: Any, # Protocol event data
        *,
        run_id: UUID, # Current LLM run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Stream event:", event) # Display the protocol event
        return # Finish handling the stream event


handler: PrintLLMHandler = PrintLLMHandler() # Create the callback handler

success_manager: CallbackManagerForLLMRun = CallbackManagerForLLMRun( # Create a successful LLM run manager
    run_id=uuid4(), # Generate the successful run identifier
    handlers=[handler], # Add the active handler
    inheritable_handlers=[handler], # Allow child runs to inherit the handler
    tags=["demo-llm"], # Add tags to the callback events
    metadata={"model": "fake-llm"}, # Store run metadata
) # Finish creating the manager

first_chunk: GenerationChunk = GenerationChunk(text="Hello ") # Create the first generation chunk

second_chunk: GenerationChunk = GenerationChunk(text="LangChain") # Create the second generation chunk

success_manager.on_llm_new_token( # Dispatch the first generated token
    "Hello ", # Supply the token text
    chunk=first_chunk, # Supply the matching generation chunk
) # Finish dispatching the token

success_manager.on_llm_new_token( # Dispatch the second generated token
    "LangChain", # Supply the token text
    chunk=second_chunk, # Supply the matching generation chunk
) # Finish dispatching the token

success_manager.on_stream_event( # Dispatch one version-3-style protocol event
    {
        "type": "message_chunk", # Describe the protocol event type
        "data": {"content": "Hello LangChain"}, # Store free-form event data
    }
) # Finish dispatching the stream event

final_result: LLMResult = LLMResult( # Create the completed LLM result
    generations=[ # Store generations for each prompt
        [
            Generation(text="Hello LangChain"), # Store the final generated text
        ]
    ],
    llm_output={"model": "fake-llm"}, # Store provider-specific output
) # Finish creating the result

success_manager.on_llm_end(final_result) # Dispatch the successful completion event


error_manager: CallbackManagerForLLMRun = CallbackManagerForLLMRun( # Create a separate failed LLM run manager
    run_id=uuid4(), # Generate the failed run identifier
    handlers=[handler], # Add the active handler
    inheritable_handlers=[handler], # Allow child runs to inherit the handler
) # Finish creating the failed manager

error_manager.on_llm_error( # Dispatch an LLM error event
    ValueError("Model generation failed") # Supply the raised exception
) # Finish dispatching the error

# `AsyncCallbackManagerForLLMRun: AsyncRunManager, LLMManagerMixin`

Asynchronous callback manager bound to one LLM or chat-model run.

## Methods

### `get_sync`

Returns an equivalent `CallbackManagerForLLMRun`.

### `on_llm_new_token`

Asynchronously dispatches a generated token or structured content block.

### `on_llm_end`

Asynchronously dispatches the completed `LLMResult`.

### `on_llm_error`

Asynchronously dispatches a generation exception.

### `on_stream_event`

Asynchronously dispatches one protocol event from `astream_events(version="v3")`.

# `CallbackManagerForChainRun: ParentRunManager, ChainManagerMixin`

Synchronous callback manager bound to one chain run.

## Methods

### `on_chain_end`

Dispatches the chain output.

### `on_chain_error`

Dispatches an exception raised by the chain.

### `on_agent_action`

Dispatches an `AgentAction`.

### `on_agent_finish`

Dispatches an `AgentFinish`.

In [ ]:
from typing import Any # Import Any for additional callback arguments
from uuid import UUID, uuid4 # Import UUID types and generator

from langchain_core.agents import AgentAction, AgentFinish # Import agent event types
from langchain_core.callbacks import BaseCallbackHandler # Import the callback-handler base class
from langchain_core.callbacks.manager import CallbackManagerForChainRun # Import the chain run manager


class PrintChainHandler(BaseCallbackHandler): # Create a handler for chain and agent events
    def on_agent_action( # Handle an agent action
        self, # Current handler instance
        action: AgentAction, # Action selected by the agent
        *,
        run_id: UUID, # Current chain run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Agent action:", action.tool, action.tool_input) # Display the selected tool and input
        return # Finish handling the event

    def on_agent_finish( # Handle the agent's final response
        self, # Current handler instance
        finish: AgentFinish, # Final agent result
        *,
        run_id: UUID, # Current chain run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Agent finish:", finish.return_values) # Display the final values
        return # Finish handling the event

    def on_chain_end( # Handle successful chain completion
        self, # Current handler instance
        outputs: dict[str, Any], # Output produced by the chain
        *,
        run_id: UUID, # Current chain run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Chain output:", outputs) # Display the chain output
        return # Finish handling the event

    def on_chain_error( # Handle a chain error
        self, # Current handler instance
        error: BaseException, # Exception raised by the chain
        *,
        run_id: UUID, # Current chain run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Chain error:", error) # Display the exception
        return # Finish handling the event


handler: PrintChainHandler = PrintChainHandler() # Create the callback handler

success_manager: CallbackManagerForChainRun = CallbackManagerForChainRun( # Create a successful chain manager
    run_id=uuid4(), # Generate the chain run identifier
    handlers=[handler], # Add the active handler
    inheritable_handlers=[handler], # Allow child runs to inherit the handler
) # Finish creating the manager

action: AgentAction = AgentAction( # Create an agent action
    tool="calculator", # Set the selected tool
    tool_input="10 + 20", # Set the input passed to the tool
    log="Using calculator", # Store the agent reasoning log
) # Finish creating the action

finish: AgentFinish = AgentFinish( # Create the final agent result
    return_values={"output": "30"}, # Store the final response
    log="Calculation completed", # Store the completion log
) # Finish creating the result

success_manager.on_agent_action(action) # Dispatch the agent-action event

success_manager.on_agent_finish(finish) # Dispatch the agent-finish event

success_manager.on_chain_end({"result": 30}) # Dispatch successful chain completion

error_manager: CallbackManagerForChainRun = CallbackManagerForChainRun( # Create a failed chain manager
    run_id=uuid4(), # Generate another run identifier
    handlers=[handler], # Add the active handler
    inheritable_handlers=[handler], # Allow child runs to inherit the handler
) # Finish creating the failed manager

error_manager.on_chain_error(ValueError("Chain execution failed")) # Dispatch a chain error

# `AsyncCallbackManagerForChainRun: AsyncParentRunManager, ChainManagerMixin`

Asynchronous callback manager bound to one chain run.

## Methods

### `get_sync`

Returns an equivalent `CallbackManagerForChainRun`.

### `on_chain_end`

Asynchronously dispatches the chain output.

### `on_chain_error`

Asynchronously dispatches a chain exception.

### `on_agent_action`

Asynchronously dispatches an `AgentAction`.

### `on_agent_finish`

Asynchronously dispatches an `AgentFinish`.

# `CallbackManagerForToolRun: ParentRunManager, ToolManagerMixin`

Synchronous callback manager bound to one tool run.

## Methods

### `on_tool_end`

Dispatches the tool output.

### `on_tool_error`

Dispatches an exception raised by the tool.

In [ ]:
from typing import Any # Import Any for additional callback arguments
from uuid import UUID, uuid4 # Import UUID types and generator

from langchain_core.callbacks import BaseCallbackHandler # Import the callback-handler base class
from langchain_core.callbacks.manager import CallbackManagerForToolRun # Import the tool run manager


class PrintToolHandler(BaseCallbackHandler): # Create a handler for tool events
    def on_tool_end( # Handle successful tool completion
        self, # Current handler instance
        output: Any, # Output returned by the tool
        *,
        run_id: UUID, # Current tool run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Tool output:", output) # Display the successful tool output
        return # Finish handling the event

    def on_tool_error( # Handle a tool execution error
        self, # Current handler instance
        error: BaseException, # Exception raised by the tool
        *,
        run_id: UUID, # Current tool run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Tool error:", error) # Display the tool exception
        return # Finish handling the event


handler: PrintToolHandler = PrintToolHandler() # Create the callback handler

success_manager: CallbackManagerForToolRun = CallbackManagerForToolRun( # Create a successful tool run manager
    run_id=uuid4(), # Generate the tool run identifier
    handlers=[handler], # Add the active callback handler
    inheritable_handlers=[handler], # Allow child runs to inherit the handler
) # Finish creating the manager

success_manager.on_tool_end("Calculation result: 30") # Dispatch successful tool completion

error_manager: CallbackManagerForToolRun = CallbackManagerForToolRun( # Create a failed tool run manager
    run_id=uuid4(), # Generate another tool run identifier
    handlers=[handler], # Add the active callback handler
    inheritable_handlers=[handler], # Allow child runs to inherit the handler
) # Finish creating the failed manager

error_manager.on_tool_error(ValueError("Invalid calculator input")) # Dispatch a tool error

# `AsyncCallbackManagerForToolRun: AsyncParentRunManager, ToolManagerMixin`

Asynchronous callback manager bound to one tool run.

## Methods

### `get_sync`

Returns an equivalent `CallbackManagerForToolRun`.

### `on_tool_end`

Asynchronously dispatches the tool output.

### `on_tool_error`

Asynchronously dispatches a tool exception.

---

# `CallbackManagerForRetrieverRun: ParentRunManager, RetrieverManagerMixin`

Synchronous callback manager bound to one retriever run.

## Methods

### `on_retriever_end`

Dispatches the retrieved documents.

### `on_retriever_error`

Dispatches an exception raised by the retriever.

# `AsyncCallbackManagerForRetrieverRun: AsyncParentRunManager, RetrieverManagerMixin`

Asynchronous callback manager bound to one retriever run.

## Methods

### `get_sync`

Returns an equivalent `CallbackManagerForRetrieverRun`.

### `on_retriever_end`

Asynchronously dispatches the retrieved documents.

### `on_retriever_error`

Asynchronously dispatches a retriever exception.

# `CallbackManager: BaseCallbackManager`

`CallbackManager` dispatches synchronous start and custom events and creates run-specific managers.

## Methods

### `on_llm_start`

Dispatches one start event per prompt and returns one `CallbackManagerForLLMRun` per prompt.

### `on_chat_model_start`

Dispatches one start event per message list and returns one `CallbackManagerForLLMRun` per input.

### `on_chain_start`

Dispatches a chain start event and returns a `CallbackManagerForChainRun`.

### `on_tool_start`

Dispatches a tool start event and returns a `CallbackManagerForToolRun`.

### `on_retriever_start`

Dispatches a retriever start event and returns a `CallbackManagerForRetrieverRun`.

### `on_custom_event`

Dispatches a named custom event with free-form data.

## Class Method

### `configure`

Creates a callback manager from inheritable and local callbacks, tags, and metadata.

```python
CallbackManager.configure(
    inheritable_callbacks: Callbacks = None, # Callbacks inherited by child runs
    local_callbacks: Callbacks = None, # Callbacks used only by the current run
    verbose: bool = False, # Whether standard verbose output is enabled
    inheritable_tags: list[str] | None = None, # Tags inherited by child runs
    local_tags: list[str] | None = None, # Tags used only by the current run
    inheritable_metadata: dict[str, Any] | None = None, # Metadata inherited by child runs
    local_metadata: dict[str, Any] | None = None, # Metadata used only by the current run
    *,
    langsmith_inheritable_metadata: Mapping[str, Any] | None = None, # Default metadata applied to LangChainTracer handlers
    langsmith_inheritable_tags: list[str] | None = None, # Default tags applied to LangChainTracer handlers
) -> CallbackManager # Return the configured manager
```

In [ ]:
from typing import Any # Import Any for flexible callback values
from uuid import UUID # Import UUID for run identifiers

from langchain_core.callbacks import BaseCallbackHandler # Import the callback-handler base class
from langchain_core.callbacks.manager import CallbackManager # Import the synchronous callback manager
from langchain_core.documents import Document # Import the document type
from langchain_core.messages import HumanMessage # Import the human-message type
from langchain_core.outputs import Generation, LLMResult # Import LLM result types


class PrintHandler(BaseCallbackHandler): # Create a handler for start and custom events
    def on_llm_start( # Handle an LLM start event
        self, # Current handler instance
        serialized: dict[str, Any], # Serialized LLM information
        prompts: list[str], # Prompts supplied to the LLM
        *,
        run_id: UUID, # Current run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("LLM started:", prompts) # Display the prompts
        return # Finish handling the event

    def on_chat_model_start( # Handle a chat-model start event
        self, # Current handler instance
        serialized: dict[str, Any], # Serialized model information
        messages: list[list[Any]], # Batched chat messages
        *,
        run_id: UUID, # Current run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Chat model started:", messages[0][0].content) # Display the first message
        return # Finish handling the event

    def on_chain_start( # Handle a chain start event
        self, # Current handler instance
        serialized: dict[str, Any], # Serialized chain information
        inputs: dict[str, Any], # Chain inputs
        *,
        run_id: UUID, # Current run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Chain started:", inputs) # Display the chain inputs
        return # Finish handling the event

    def on_tool_start( # Handle a tool start event
        self, # Current handler instance
        serialized: dict[str, Any], # Serialized tool information
        input_str: str, # Tool input
        *,
        run_id: UUID, # Current run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Tool started:", input_str) # Display the tool input
        return # Finish handling the event

    def on_retriever_start( # Handle a retriever start event
        self, # Current handler instance
        serialized: dict[str, Any], # Serialized retriever information
        query: str, # Retrieval query
        *,
        run_id: UUID, # Current run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Retriever started:", query) # Display the query
        return # Finish handling the event

    def on_custom_event( # Handle a custom callback event
        self, # Current handler instance
        name: str, # Custom event name
        data: Any, # Custom event data
        *,
        run_id: UUID, # Current run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Custom event:", name, data) # Display the custom event
        return # Finish handling the event


handler: PrintHandler = PrintHandler() # Create the callback handler

manager: CallbackManager = CallbackManager.configure( # Configure the callback manager
    inheritable_callbacks=[handler], # Add a handler inherited by child runs
    inheritable_tags=["demo"], # Add an inheritable tag
    inheritable_metadata={"source": "example"}, # Add inheritable metadata
) # Finish configuring the manager

llm_runs = manager.on_llm_start( # Dispatch LLM start events
    {"name": "fake-llm"}, # Supply serialized LLM information
    ["Explain Python"], # Supply one prompt
) # Receive one LLM run manager per prompt

llm_runs[0].on_llm_end( # Complete the LLM run
    LLMResult(generations=[[Generation(text="Python explanation")]]) # Supply the LLM result
) # Finish the LLM run

chat_runs = manager.on_chat_model_start( # Dispatch chat-model start events
    {"name": "fake-chat-model"}, # Supply serialized model information
    [[HumanMessage(content="Hello")]], # Supply one list of messages
) # Receive one LLM run manager per message list

chain_run = manager.on_chain_start( # Dispatch a chain start event
    {"name": "demo-chain"}, # Supply serialized chain information
    {"number": 10}, # Supply chain inputs
) # Receive a chain run manager

chain_run.on_chain_end({"result": 20}) # Complete the chain run

tool_run = manager.on_tool_start( # Dispatch a tool start event
    {"name": "calculator"}, # Supply serialized tool information
    "10 + 20", # Supply the tool input
) # Receive a tool run manager

tool_run.on_tool_end("30") # Complete the tool run

retriever_run = manager.on_retriever_start( # Dispatch a retriever start event
    {"name": "document-search"}, # Supply serialized retriever information
    "LangChain", # Supply the retrieval query
) # Receive a retriever run manager

retriever_run.on_retriever_end( # Complete the retriever run
    [Document(page_content="LangChain helps build LLM applications.")] # Supply retrieved documents
) # Finish the retriever run

manager.on_custom_event( # Dispatch a free-form custom event
    "progress_update", # Set the custom event name
    {"percentage": 100}, # Supply custom event data
) # Finish dispatching the custom event

# `CallbackManagerForChainGroup: CallbackManager`

Synchronous callback manager returned by `trace_as_chain_group()`.

## Fields

```python
parent_run_manager: CallbackManagerForChainRun # Run manager representing the traced group
ended: bool # Whether the group has already ended
```

## Constructor

```python
CallbackManagerForChainGroup(
    handlers: list[BaseCallbackHandler], # Active callback handlers
    inheritable_handlers: list[BaseCallbackHandler] | None = None, # Handlers inherited by child runs
    parent_run_id: UUID | None = None, # Optional parent run identifier
    *,
    parent_run_manager: CallbackManagerForChainRun, # Run manager representing the group
    **kwargs: Any, # Additional callback-manager fields
) -> None # Initialize the chain-group manager
```

## Overridden Methods

### `copy`

Returns a copy while preserving the parent run manager.

### `merge`

Merges handlers, tags, and metadata while preserving the current parent run manager.

### `on_chain_end`

Marks the group as ended and forwards the output to the parent run manager.

### `on_chain_error`

Marks the group as ended and forwards the exception to the parent run manager.

In [ ]:
from typing import Any # Import Any for callback values
from uuid import UUID, uuid4 # Import UUID types and generator

from langchain_core.callbacks import BaseCallbackHandler # Import the callback-handler base class
from langchain_core.callbacks.manager import ( # Import chain callback managers
    CallbackManager, # Import the normal callback manager
    CallbackManagerForChainGroup, # Import the chain-group callback manager
    CallbackManagerForChainRun, # Import the parent chain-run manager
) # Finish importing callback managers


class PrintHandler(BaseCallbackHandler): # Create a handler for group completion and errors
    def on_chain_end( # Handle successful group completion
        self, # Current handler instance
        outputs: Any, # Output produced by the chain group
        *,
        run_id: UUID, # Parent chain-run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Group output:", outputs) # Display the group output
        return # Finish handling the event

    def on_chain_error( # Handle a chain-group error
        self, # Current handler instance
        error: BaseException, # Exception raised by the chain group
        *,
        run_id: UUID, # Parent chain-run identifier
        **kwargs: Any, # Additional callback information
    ) -> None: # Return no value
        print("Group error:", error) # Display the group error
        return # Finish handling the event


handler: PrintHandler = PrintHandler() # Create the callback handler

parent_run: CallbackManagerForChainRun = CallbackManagerForChainRun( # Create the parent chain run
    run_id=uuid4(), # Generate the parent run identifier
    handlers=[handler], # Add the active handler
    inheritable_handlers=[handler], # Allow child runs to inherit the handler
) # Finish creating the parent run

group_manager: CallbackManagerForChainGroup = CallbackManagerForChainGroup( # Create the chain-group manager
    handlers=[handler], # Add the active handler
    inheritable_handlers=[handler], # Add inherited handlers
    parent_run_id=parent_run.run_id, # Link the manager to the parent run
    parent_run_manager=parent_run, # Store the parent chain-run manager
    tags=["group"], # Add a group tag
    metadata={"type": "demo"}, # Add group metadata
) # Finish creating the group manager

copied_manager: CallbackManagerForChainGroup = group_manager.copy() # Copy the group manager

extra_manager: CallbackManager = CallbackManager.configure( # Create another callback manager
    local_tags=["extra"], # Add another tag
    local_metadata={"source": "example"}, # Add additional metadata
) # Finish creating the extra manager

merged_manager: CallbackManagerForChainGroup = group_manager.merge( # Merge callback-manager state
    extra_manager # Supply the manager to merge
) # Finish merging the managers

print("Initially ended:", group_manager.ended) # Display the initial group state

print("Copy keeps parent:", copied_manager.parent_run_manager is parent_run) # Verify the copied parent manager

print("Merged tags:", merged_manager.tags) # Display merged tags

group_manager.on_chain_end({"result": 30}) # End the group successfully

print("Ended after success:", group_manager.ended) # Display the updated group state


error_group: CallbackManagerForChainGroup = CallbackManagerForChainGroup( # Create another group for an error
    handlers=[handler], # Add the active handler
    inheritable_handlers=[handler], # Add inherited handlers
    parent_run_id=parent_run.run_id, # Link it to the parent run
    parent_run_manager=parent_run, # Store the parent manager
) # Finish creating the error group

error_group.on_chain_error(ValueError("Group execution failed")) # End the group with an error

print("Error group ended:", error_group.ended) # Display the error-group state

# `AsyncCallbackManager: BaseCallbackManager`

`AsyncCallbackManager` dispatches asynchronous start and custom events and creates asynchronous run-specific managers.

## Property

### `is_async`

Returns `True`.

## Methods

### `on_llm_start`

Asynchronously dispatches one start event per prompt and returns one `AsyncCallbackManagerForLLMRun` per prompt.

### `on_chat_model_start`

Asynchronously dispatches one start event per message list and returns one `AsyncCallbackManagerForLLMRun` per input.

### `on_chain_start`

Asynchronously dispatches a chain start event and returns an `AsyncCallbackManagerForChainRun`.

### `on_tool_start`

Asynchronously dispatches a tool start event and returns an `AsyncCallbackManagerForToolRun`.

### `on_retriever_start`

Asynchronously dispatches a retriever start event and returns an `AsyncCallbackManagerForRetrieverRun`.

### `on_custom_event`

Asynchronously dispatches a named custom event with free-form data.

## Class Method

### `configure`

Creates an asynchronous callback manager from inheritable and local callbacks, tags, and metadata.

```python
AsyncCallbackManager.configure(
    inheritable_callbacks: Callbacks = None, # Callbacks inherited by child runs
    local_callbacks: Callbacks = None, # Callbacks used only by the current run
    verbose: bool = False, # Whether standard verbose output is enabled
    inheritable_tags: list[str] | None = None, # Tags inherited by child runs
    local_tags: list[str] | None = None, # Tags used only by the current run
    inheritable_metadata: dict[str, Any] | None = None, # Metadata inherited by child runs
    local_metadata: dict[str, Any] | None = None, # Metadata used only by the current run
    *,
    langsmith_inheritable_metadata: Mapping[str, Any] | None = None, # Default metadata applied to LangChainTracer handlers
    langsmith_inheritable_tags: list[str] | None = None, # Default tags applied to LangChainTracer handlers
) -> AsyncCallbackManager # Return the configured asynchronous manager
```

# `AsyncCallbackManagerForChainGroup: AsyncCallbackManager`

Asynchronous callback manager returned by `atrace_as_chain_group()`.

## Fields

```python
parent_run_manager: AsyncCallbackManagerForChainRun # Async run manager representing the traced group
ended: bool # Whether the group has already ended
```

## Constructor

```python
AsyncCallbackManagerForChainGroup(
    handlers: list[BaseCallbackHandler], # Active callback handlers
    inheritable_handlers: list[BaseCallbackHandler] | None = None, # Handlers inherited by child runs
    parent_run_id: UUID | None = None, # Optional parent run identifier
    *,
    parent_run_manager: AsyncCallbackManagerForChainRun, # Async run manager representing the group
    **kwargs: Any, # Additional callback-manager fields
) -> None # Initialize the asynchronous chain-group manager
```

## Overridden Methods

### `copy`

Returns a copy while preserving the parent run manager.

### `merge`

Merges handlers, tags, and metadata while preserving the current parent run manager.

### `on_chain_end`

Marks the group as ended and asynchronously forwards the output.

### `on_chain_error`

Marks the group as ended and asynchronously forwards the exception.

---

# `dispatch_custom_event`

Dispatches a synchronous custom event from inside an existing Runnable or tool run.

## Syntax

```python
dispatch_custom_event(
    name: str, # Custom event name
    data: Any, # Free-form event data
    *,
    config: RunnableConfig | None = None, # Optional Runnable configuration
) -> None # Dispatch the event
```

Raises `RuntimeError` when no parent run is available.

---

# `adispatch_custom_event`

Asynchronously dispatches a custom event from inside an existing Runnable or tool run.

## Syntax

```python
async adispatch_custom_event(
    name: str, # Custom event name
    data: Any, # Free-form event data
    *,
    config: RunnableConfig | None = None, # Optional Runnable configuration
) -> None # Dispatch the event asynchronously
```

Raises `RuntimeError` when no parent run is available.

On Python 3.10, pass `config` explicitly when asynchronous Runnable configuration cannot be propagated automatically.